# NicoComic YOLOX-Nano 512 panel training

Use a free Google Colab GPU runtime. This run downloads only the SHA-pinned CC0-1.0 Comix v0 tiny dataset and the public model repository. Faster R-CNN boxes are pseudo labels for training; they are not product quality truth. No private comics or annotations are uploaded.


In [ ]:
!nvidia-smi
import torch
assert torch.cuda.is_available(), 'Select Runtime > Change runtime type > GPU'


In [ ]:
%%bash
set -euo pipefail
if [ ! -d /content/NicoComic-Reader-Models/.git ]; then
  git clone -q https://github.com/zyuanming/NicoComic-Reader-Models.git /content/NicoComic-Reader-Models
fi
cd /content/NicoComic-Reader-Models
if [ ! -d /content/YOLOX/.git ]; then
  git clone -q https://github.com/Megvii-BaseDetection/YOLOX.git /content/YOLOX
fi
git -C /content/YOLOX checkout -q 419778480ab6ec0590e5d3831b3afb3b46ab2aa3
python -m pip install -q -r scripts/requirements-training.txt
python -m pip install -q -e /content/YOLOX --no-build-isolation --no-deps


In [ ]:
%%bash
set -euo pipefail
cd /content/NicoComic-Reader-Models
python scripts/download_comix_v0.py datasets/comix_v0_tiny_pages.json /content/comix-v0
if [ ! -f /content/comix-v0-coco/annotations/instances_train2017.json ] || [ ! -f /content/comix-v0-coco/annotations/instances_val2017.json ]; then
  python scripts/prepare_comix_v0_coco.py datasets/comix_v0_tiny_pages.json /content/comix-v0-coco /content/comix-v0/*.tar
fi


In [ ]:
%%bash
set -euo pipefail
cd /content/NicoComic-Reader-Models
python scripts/train_yolox_panels.py experiments/yolox_nano_panels_512.py /content/comix-v0-coco /content/yolox-512 --device cuda --batch-size 8 --epochs 100 --workers 2 --resume
zip -q -j /content/yolox-512-checkpoints.zip /content/yolox-512/epoch_*_ckpt.pth


In [ ]:
%%bash
set -euo pipefail
cd /content/NicoComic-Reader-Models
python scripts/evaluate_yolox_coco.py experiments/yolox_nano_panels_512.py /content/yolox-512/latest_ckpt.pth /content/comix-v0-coco/annotations/instances_val2017.json /content/comix-v0-coco/val2017 /content/yolox-512-validation.json
cat /content/yolox-512-validation.json


In [ ]:
import subprocess
subprocess.run(['python', 'scripts/export_yolox_onnx.py', 'experiments/yolox_nano_panels_512.py', '/content/yolox-512/latest_ckpt.pth', '/content/NicoComicPanelYOLOXNano512.onnx'], cwd='/content/NicoComic-Reader-Models', check=True)
from google.colab import files
files.download('/content/NicoComicPanelYOLOXNano512.onnx')
files.download('/content/yolox-512-validation.json')
files.download('/content/yolox-512-checkpoints.zip')


After downloading the ONNX file, convert it to Core ML on macOS and run the unchanged private 60-panel/20-fallback scorer. Do not publish or integrate the model based on the pseudo-label validation score.
